# Notebook 4 — Clasificador BiLSTM → MongoDB
### Investing AI · iDeSo · UNMSM

**Objetivo:** Leer datos de `precios_ohlcv` en MongoDB, construir secuencias temporales,
entrenar un clasificador **BiLSTM** (Bidirectional LSTM) para predecir señales BUY/SELL,
guardar predicciones y métricas en `predicciones`/`metricas_modelos`, y **comparar**
sus resultados contra el LSTM unidireccional del Notebook 3.

**Arquitectura:** `Bidirectional(LSTM(200, return_sequences=True)) → Dropout(0.3) → Bidirectional(LSTM(100)) → Dense(1, sigmoid)`
**Entrenamiento:** 80 épocas · batch_size=64 · Adam(lr=0.001) · binary_crossentropy

**Prerequisitos:**
- Notebook 1 (Ingesta) ejecutado — colección `precios_ohlcv` poblada.
- Notebook 3 (LSTM) ejecutado — colección `metricas_modelos` con `modelo='LSTM'`, usado como referencia de comparación.

**Salida:** Colecciones `predicciones`/`metricas_modelos` con `modelo='BiLSTM'`, y una
tabla comparativa BiLSTM vs. LSTM al final del notebook.

> Inspirado en la investigación doctoral del Prof. Cancho-Rodríguez sobre redes neuronales recurrentes bidireccionales aplicadas a series financieras.

## Paso 1 — Instalación de librerías

In [ ]:
# Instalación de librerías necesarias
# TensorFlow ya viene preinstalado en Colab; se asegura scikit-learn y pymongo
!pip install 'pymongo[srv]' scikit-learn --quiet
print('✓ Librerías instaladas')

## Paso 2 — Conexión a MongoDB Atlas

In [ ]:
# Conectar a MongoDB Atlas usando el Secret MONGO_URI de Colab
# (Secrets: ícono de llave en el panel izquierdo de Colab)
from google.colab import userdata
from pymongo import MongoClient
import pandas as pd
import numpy as np
from datetime import datetime
import json
import random

MONGO_URI = userdata.get('MONGO_URI')
client    = MongoClient(MONGO_URI)
db        = client['spbi']

# Verificar conexión
client.admin.command('ping')
print('✓ Conectado a MongoDB Atlas')
print(f'  Base de datos: spbi')
print(f'  Colecciones disponibles: {db.list_collection_names()}')

## Paso 3 — Semilla fija y configuración de reproducibilidad

In [ ]:
# SEED fija para reproducibilidad (numpy, python, tensorflow)
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'✓ Semilla fija SEED={SEED} aplicada (random, numpy, tensorflow)')
print(f'  TensorFlow versión: {tf.__version__}')

## Paso 4 — Carga de datos desde MongoDB

In [ ]:
# Función para cargar precios OHLCV de un ticker desde MongoDB
def cargar_datos(ticker):
    """
    Lee los documentos de precios_ohlcv para un ticker
    y los devuelve como DataFrame ordenado por fecha.
    """
    docs = list(db['precios_ohlcv'].find({'ticker': ticker}).sort('fecha', 1))
    if not docs:
        return pd.DataFrame()
    df = pd.DataFrame(docs)
    df['fecha'] = pd.to_datetime(df['fecha'])
    df = df.sort_values('fecha').reset_index(drop=True)
    return df

# Verificación rápida con BVN
df_test = cargar_datos('BVN')
print(f'✓ BVN: {len(df_test)} registros cargados')
print(df_test[['fecha','close','sma_20','rsi_14']].tail(3).to_string(index=False))

## Paso 5 — Ingeniería de features, secuencias temporales y target

Misma lógica de feature engineering y de construcción de secuencias que el Notebook 3 (LSTM),
para que la comparación entre arquitecturas sea justa: **mismos features, mismo `N_STEPS`,
misma partición temporal 80/20** — la única variable que cambia es la arquitectura de la red.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

N_STEPS = 20  # tamaño de la ventana temporal (días), igual que en el Notebook 3 (LSTM)

# Features para tickers con precio variable (FSM, ABX.TO, BVN, BHP)
FEATURES_PRECIO = ['close','sma_20','sma_50','ema_12','ema_26','rsi_14','retorno']

# Features especiales para VOLCABC1.LM (precio constante → usar volumen)
FEATURES_VOLUMEN = ['vol_change_pct','vol_relative','vol_rsi','obv_slope','money_flow_rel']

def preparar_features_precio(df):
    """
    Feature engineering estándar basado en precio e indicadores técnicos.
    Target: 1 (BUY) si el cierre del día siguiente es mayor, 0 (SELL) si baja.
    """
    df = df.copy()
    df['retorno'] = df['close'].pct_change()
    # Target: ¿sube mañana?
    df['target'] = (df['close'].shift(-1) > df['close']).astype(int)
    df = df.dropna(subset=FEATURES_PRECIO + ['target'])

    feats   = df[FEATURES_PRECIO].values
    target  = df['target'].values
    fechas  = df['fecha'].values
    precios = df['close'].values
    return feats, target, fechas, precios

def preparar_features_volumen(df):
    """
    Feature engineering basado en VOLUMEN para VOLCABC1.LM.
    Usado porque el precio de ese ticker es casi constante y los
    indicadores técnicos estándar no aportan señal estadística útil.
    """
    df = df.copy()
    df['volume'] = pd.to_numeric(df['volume'], errors='coerce').fillna(0)

    # Cambio porcentual diario de volumen
    df['vol_change_pct'] = df['volume'].pct_change().fillna(0).clip(-5, 5)
    # Volumen relativo respecto a su media de 20 días
    df['vol_sma20']      = df['volume'].rolling(20).mean()
    df['vol_relative']   = (df['volume'] / df['vol_sma20'].replace(0, np.nan)).fillna(1).clip(0, 10)
    # OBV (On-Balance Volume)
    direction    = np.sign(df['close'].diff().fillna(0))
    df['obv']    = (direction * df['volume']).cumsum()
    df['obv_sma20']  = df['obv'].rolling(20).mean()
    df['obv_slope']  = df['obv'].diff(5).fillna(0)
    # RSI aplicado al volumen
    delta_vol = df['volume'].diff()
    gain_vol  = delta_vol.clip(lower=0).rolling(14).mean()
    loss_vol  = (-delta_vol.clip(upper=0)).rolling(14).mean()
    rs_vol    = gain_vol / loss_vol.replace(0, np.nan)
    df['vol_rsi'] = (100 - 100 / (1 + rs_vol)).fillna(50)
    # Money flow relativo
    df['money_flow']     = df['close'] * df['volume']
    df['mf_sma10']       = df['money_flow'].rolling(10).mean()
    df['money_flow_rel'] = (df['money_flow'] / df['mf_sma10'].replace(0, np.nan)).fillna(1)

    # Target: señal basada en OBV vs su media móvil (sin filtro de magnitud)
    # Nota: igual que en el Notebook 3, NO se descartan filas por
    # 'vol_relative > 1.3', porque el BiLSTM también necesita tramos de
    # días CONSECUTIVOS para poder armar secuencias temporales.
    df['target'] = (df['obv'] > df['obv_sma20']).astype(int)
    df = df.dropna(subset=FEATURES_VOLUMEN)

    feats   = df[FEATURES_VOLUMEN].values
    target  = df['target'].values
    fechas  = df['fecha'].values
    precios = df['close'].values
    return feats, target, fechas, precios

def crear_secuencias(feats, target, fechas, precios, n_steps=N_STEPS):
    """
    Convierte una matriz de features 2D (muestras, features) en
    secuencias 3D (muestras, n_steps, features) mediante ventana deslizante.
    El target de cada secuencia es el target del último día de la ventana.
    """
    X_seq, y_seq, fechas_seq, precios_seq = [], [], [], []
    for i in range(n_steps, len(feats)):
        X_seq.append(feats[i - n_steps:i])
        y_seq.append(target[i])
        fechas_seq.append(fechas[i])
        precios_seq.append(precios[i])
    return (np.array(X_seq), np.array(y_seq),
            np.array(fechas_seq), np.array(precios_seq))

print(f'✓ Funciones de feature engineering y secuencias definidas (N_STEPS={N_STEPS})')

## Paso 6 — Construcción del modelo BiLSTM (Keras Sequential)

**Diferencia clave respecto al LSTM del Notebook 3:** cada capa `LSTM` va envuelta en
`Bidirectional(...)`, lo que hace que la red procese la secuencia **en ambas direcciones**
(pasado→futuro y futuro→pasado dentro de la ventana de `N_STEPS` días) y concatene ambas
representaciones. Esto duplica efectivamente el número de unidades de salida por capa,
por lo que la capacidad del modelo es mayor que la del LSTM unidireccional equivalente.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

def construir_modelo_bilstm(n_steps, n_features):
    """
    Arquitectura BiLSTM para clasificación binaria BUY/SELL:
    Bidirectional(LSTM(200, return_sequences=True)) -> Dropout(0.3) ->
    Bidirectional(LSTM(100)) -> Dense(1, sigmoid)
    """
    modelo = Sequential([
        Input(shape=(n_steps, n_features)),
        Bidirectional(LSTM(200, return_sequences=True)),
        Dropout(0.3),
        Bidirectional(LSTM(100)),
        Dense(1, activation='sigmoid')
    ])

    modelo.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return modelo

print('✓ Función construir_modelo_bilstm definida')
print('  Arquitectura: Bidirectional(LSTM(200, return_sequences=True)) -> Dropout(0.3) ->')
print('                Bidirectional(LSTM(100)) -> Dense(1, sigmoid)')

## Paso 7 — Entrenamiento y evaluación del BiLSTM

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score, confusion_matrix)

def entrenar_bilstm(X, y):
    """
    Entrena el BiLSTM sobre una partición temporal 80/20 (sin data leakage).
    Misma metodología que entrenar_lstm() del Notebook 3, para que la
    comparación entre arquitecturas sea justa:
    - MinMaxScaler ajustado SOLO con datos de entrenamiento
    - 80 épocas, batch_size=64, Adam(lr=0.001)
    - class_weight balanceado
    - EarlyStopping sobre val_loss para evitar sobreajuste excesivo
    """
    n     = len(X)
    corte = int(n * 0.80)

    n_steps, n_features = X.shape[1], X.shape[2]

    # Escalado: el scaler se ajusta sobre los datos de train aplanados (2D)
    scaler = MinMaxScaler()
    X_train_flat = X[:corte].reshape(-1, n_features)
    scaler.fit(X_train_flat)

    def escalar(X_):
        forma = X_.shape
        X_flat = X_.reshape(-1, n_features)
        X_esc  = scaler.transform(X_flat)
        return X_esc.reshape(forma)

    X_train, X_test = escalar(X[:corte]), escalar(X[corte:])
    y_train, y_test = y[:corte], y[corte:]

    # Pesos de clase para corregir desbalance (equivalente a class_weight='balanced')
    clases, conteos = np.unique(y_train, return_counts=True)
    total = len(y_train)
    pesos_clase = {int(c): total / (len(clases) * cnt) for c, cnt in zip(clases, conteos)}

    modelo = construir_modelo_bilstm(n_steps, n_features)

    early_stop = EarlyStopping(
        monitor='val_loss', patience=10,
        restore_best_weights=True, verbose=0
    )

    historial = modelo.fit(
        X_train, y_train,
        epochs=80,
        batch_size=64,
        validation_split=0.15,
        class_weight=pesos_clase,
        callbacks=[early_stop],
        verbose=0
    )

    y_prob = modelo.predict(X_test, verbose=0).flatten()
    y_pred = (y_prob >= 0.5).astype(int)

    metricas = {
        'accuracy' : round(float(accuracy_score(y_test,  y_pred)), 4),
        'precision': round(float(precision_score(y_test, y_pred, average='weighted', zero_division=0)), 4),
        'recall'   : round(float(recall_score(y_test,    y_pred, average='weighted', zero_division=0)), 4),
        'f1'       : round(float(f1_score(y_test,        y_pred, average='weighted', zero_division=0)), 4),
        'epocas_entrenadas': int(len(historial.history['loss'])),
        'n_train'  : int(len(X_train)),
        'n_test'   : int(len(X_test)),
        'n_parametros': int(modelo.count_params())
    }

    cm = confusion_matrix(y_test, y_pred).tolist()

    hist_epocas = {
        'loss'         : [round(float(v), 4) for v in historial.history['loss']],
        'accuracy'     : [round(float(v), 4) for v in historial.history['accuracy']],
        'val_loss'     : [round(float(v), 4) for v in historial.history.get('val_loss', [])],
        'val_accuracy' : [round(float(v), 4) for v in historial.history.get('val_accuracy', [])]
    }

    return modelo, scaler, metricas, cm, hist_epocas, X_test, y_test

print('✓ Función entrenar_bilstm definida')
print('  - Misma metodología que el LSTM del Notebook 3 (comparación justa)')
print('  - class_weight balanceado, MinMaxScaler solo con train, EarlyStopping patience=10')

## Paso 8 — Pipeline completo por ticker

In [ ]:
def procesar_ticker(ticker):
    """
    Pipeline completo para un ticker:
    1. Carga datos de MongoDB
    2. Selecciona feature engineering (precio o volumen según ticker)
    3. Construye secuencias temporales de tamaño N_STEPS
    4. Entrena BiLSTM sobre partición temporal 80/20
    5. Genera historial de señales sobre todas las secuencias
    6. Guarda predicción actual + métricas + historial en MongoDB
    """
    print(f'\n  [{ticker}] Procesando...')

    df = cargar_datos(ticker)
    if len(df) < 100:
        print(f'  [{ticker}] ✗ Datos insuficientes ({len(df)} registros) — se omite')
        return None

    # VOLCABC1.LM usa features de volumen (precio casi constante)
    if ticker == 'VOLCABC1.LM':
        feats, target, fechas, precios = preparar_features_volumen(df)
        tipo_features = 'volumen'
    else:
        feats, target, fechas, precios = preparar_features_precio(df)
        tipo_features = 'precio'

    X, y, fechas_seq, precios_seq = crear_secuencias(feats, target, fechas, precios, N_STEPS)

    if len(X) < 50:
        print(f'  [{ticker}] ✗ Secuencias insuficientes tras limpieza — se omite')
        return None

    print(f'  [{ticker}] Features: {tipo_features} | Secuencias: {len(X)} | N_STEPS={N_STEPS}')

    modelo, scaler, metricas, cm, hist_epocas, X_test, y_test = entrenar_bilstm(X, y)

    # ── Señal actual (última secuencia disponible) ──────────────
    n_features   = X.shape[2]
    X_esc        = scaler.transform(X.reshape(-1, n_features)).reshape(X.shape)
    ultima_X     = X_esc[-1:]
    prob_actual  = float(modelo.predict(ultima_X, verbose=0)[0][0])
    pred_actual  = int(prob_actual >= 0.5)
    senal_actual = 'BUY' if pred_actual == 1 else 'SELL'

    # ── Historial de señales (todas las secuencias) ──────────────
    probas_all = modelo.predict(X_esc, verbose=0).flatten()
    preds_all  = (probas_all >= 0.5).astype(int)

    historico_senales = [
        {
            'fecha'       : str(fechas_seq[i])[:10],
            'precio'      : round(float(precios_seq[i]), 4),
            'prediccion'  : 'BUY' if preds_all[i] == 1 else 'SELL',
            'probabilidad': round(float(probas_all[i]), 4)
        }
        for i in range(len(preds_all))
    ]

    # ── Guardar predicción actual en MongoDB ─────────────────────
    db['predicciones'].delete_many({'ticker': ticker, 'modelo': 'BiLSTM'})
    db['predicciones'].insert_one({
        'ticker'           : ticker,
        'modelo'           : 'BiLSTM',
        'senal'            : senal_actual,
        'confianza'        : round(prob_actual if pred_actual == 1 else 1 - prob_actual, 4),
        'fecha_prediccion' : datetime.now().strftime('%Y-%m-%d'),
        'historico_senales': historico_senales,
        'tipo_features'    : tipo_features,
        'n_steps'          : N_STEPS,
        'created_at'       : datetime.now()
    })

    # ── Guardar métricas en MongoDB ───────────────────────────────
    db['metricas_modelos'].delete_many({'ticker': ticker, 'modelo': 'BiLSTM'})
    db['metricas_modelos'].insert_one({
        'ticker'             : ticker,
        'modelo'             : 'BiLSTM',
        **metricas,
        'matriz_confusion'   : cm,
        'historial_epocas'   : hist_epocas,
        'tipo_features'      : tipo_features,
        'n_steps'            : N_STEPS,
        'fecha_entrenamiento': datetime.now()
    })

    a  = metricas['accuracy']
    f1 = metricas['f1']
    ep = metricas['epocas_entrenadas']
    print(f'  [{ticker}] ✓ senal={senal_actual} ({max(prob_actual, 1-prob_actual):.0%}) | acc={a:.0%} | f1={f1:.0%} | épocas={ep}')

    return {
        'ticker': ticker, 'tipo_features': tipo_features,
        'senal': senal_actual, 'metricas': metricas,
        'matriz_confusion': cm, 'historial_epocas': hist_epocas,
        'historico_senales': historico_senales
    }

print('✓ Función procesar_ticker definida')

## Paso 9 — Ejecutar pipeline para los 5 tickers

In [ ]:
TICKERS = ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']

print('=' * 55)
print('  CLASIFICADOR BiLSTM — ENTRENAMIENTO PARA 5 TICKERS')
print('=' * 55)

resultados = {}
for ticker in TICKERS:
    try:
        res = procesar_ticker(ticker)
        if res is not None:
            resultados[ticker] = res
    except Exception as e:
        print(f'  [{ticker}] ✗ Error inesperado: {e}')

print()
print('=' * 55)
print(f'  ✅ Pipeline completo: {len(resultados)}/{len(TICKERS)} tickers procesados')
print('=' * 55)

## Paso 10 — Verificación en MongoDB

In [ ]:
print('PREDICCIONES BiLSTM en MongoDB')
print('-' * 45)
for doc in db['predicciones'].find({'modelo': 'BiLSTM'}, {'_id':0,'historico_senales':0,'created_at':0}):
    t  = doc['ticker']
    s  = doc['senal']
    c  = doc['confianza']
    tf = doc.get('tipo_features','precio')
    print(f'  {t:<15} {s:<5} ({c:.0%}) | features: {tf}')

print()
print('MÉTRICAS BiLSTM en MongoDB')
print('-' * 45)
for doc in db['metricas_modelos'].find({'modelo': 'BiLSTM'}, {'_id':0,'matriz_confusion':0,'historial_epocas':0,'fecha_entrenamiento':0}):
    t  = doc['ticker']
    a  = doc['accuracy']
    f1 = doc['f1']
    p  = doc['precision']
    r  = doc['recall']
    ep = doc.get('epocas_entrenadas','?')
    npar = doc.get('n_parametros','?')
    print(f'  {t:<15} acc={a:.0%} | f1={f1:.0%} | prec={p:.0%} | rec={r:.0%} | épocas={ep} | params={npar}')

print()
print('✅ Colecciones predicciones y metricas_modelos (BiLSTM) verificadas')

## Paso 11 — Comparación BiLSTM vs. LSTM

Se leen las métricas del LSTM (`modelo='LSTM'`, guardadas por el Notebook 3) y se comparan
lado a lado contra el BiLSTM recién entrenado, ticker por ticker. Esto responde la pregunta
de diseño: **¿la bidireccionalidad mejora los resultados sobre este conjunto de datos?**

In [ ]:
print('=' * 78)
print('  COMPARACIÓN: BiLSTM vs. LSTM (unidireccional, Notebook 3)')
print('=' * 78)
print(f"  {'Ticker':<14} {'Acc LSTM':>9} {'Acc BiLSTM':>11} {'Δ Acc':>8} | {'F1 LSTM':>8} {'F1 BiLSTM':>10} {'Δ F1':>7}")
print('-' * 78)

comparacion = {}
mejoras_acc, mejoras_f1 = [], []

for ticker in TICKERS:
    doc_lstm   = db['metricas_modelos'].find_one({'ticker': ticker, 'modelo': 'LSTM'})
    doc_bilstm = db['metricas_modelos'].find_one({'ticker': ticker, 'modelo': 'BiLSTM'})

    if not doc_lstm or not doc_bilstm:
        print(f'  {ticker:<14} — Falta modelo LSTM y/o BiLSTM para este ticker, se omite')
        continue

    acc_l, acc_b = doc_lstm['accuracy'], doc_bilstm['accuracy']
    f1_l,  f1_b  = doc_lstm['f1'],       doc_bilstm['f1']
    d_acc, d_f1  = acc_b - acc_l, f1_b - f1_l

    mejoras_acc.append(d_acc)
    mejoras_f1.append(d_f1)

    comparacion[ticker] = {
        'accuracy_lstm': acc_l, 'accuracy_bilstm': acc_b, 'delta_accuracy': round(d_acc, 4),
        'f1_lstm': f1_l, 'f1_bilstm': f1_b, 'delta_f1': round(d_f1, 4),
        'n_parametros_lstm': doc_lstm.get('n_parametros'), 'n_parametros_bilstm': doc_bilstm.get('n_parametros')
    }

    flecha_acc = '▲' if d_acc > 0 else ('▼' if d_acc < 0 else '=')
    flecha_f1  = '▲' if d_f1  > 0 else ('▼' if d_f1  < 0 else '=')
    print(f"  {ticker:<14} {acc_l:>8.1%} {acc_b:>10.1%} {flecha_acc}{abs(d_acc):>6.1%} | {f1_l:>7.1%} {f1_b:>9.1%} {flecha_f1}{abs(d_f1):>5.1%}")

print('-' * 78)
if mejoras_acc:
    prom_delta_acc = sum(mejoras_acc) / len(mejoras_acc)
    prom_delta_f1  = sum(mejoras_f1)  / len(mejoras_f1)
    n_mejora_acc   = sum(1 for d in mejoras_acc if d > 0)
    n_mejora_f1    = sum(1 for d in mejoras_f1  if d > 0)

    print(f'  Δ Accuracy promedio : {prom_delta_acc:+.2%}  ({n_mejora_acc}/{len(mejoras_acc)} tickers mejoran con BiLSTM)')
    print(f'  Δ F1 promedio       : {prom_delta_f1:+.2%}  ({n_mejora_f1}/{len(mejoras_f1)} tickers mejoran con BiLSTM)')
    print()

    if prom_delta_acc > 0.01 and prom_delta_f1 > 0.01:
        conclusion = ('La bidireccionalidad SÍ aporta una mejora consistente en este dataset: '
                       'el contexto "futuro→pasado" dentro de cada ventana de N_STEPS días '
                       'ayuda al modelo a capturar patrones que el LSTM unidireccional no ve.')
    elif prom_delta_acc < -0.01 and prom_delta_f1 < -0.01:
        conclusion = ('La bidireccionalidad NO mejora los resultados en este dataset — el BiLSTM '
                       'tiene más del doble de parámetros que el LSTM, y con series financieras '
                       'relativamente cortas (1 año de datos) puede sobreajustar en vez de generalizar mejor.')
    else:
        conclusion = ('Los resultados son mixtos: la bidireccionalidad ayuda en algunos tickers '
                       'y no en otros. Esto es consistente con la naturaleza ruidosa de las series '
                       'financieras — no hay garantía de que más capacidad de modelo se traduzca '
                       'en mejor generalización fuera de muestra.')

    print('  Conclusión:')
    print(f'  {conclusion}')
else:
    conclusion = 'No hay suficientes tickers con ambos modelos entrenados para comparar.'
    print(f'  {conclusion}')

print('=' * 78)

## Paso 12 — Exportación a JSON para el frontend (`datos_bilstm_clf.json` + comparación)

In [ ]:
# Estructura de exportación, en el mismo formato que datos_lstm_clf.json del Notebook 3,
# más una sección de comparación BiLSTM vs LSTM.
export_data = {
    'modelo'        : 'BiLSTM',
    'arquitectura'  : 'Bidirectional(LSTM(200, return_sequences=True)) -> Dropout(0.3) -> Bidirectional(LSTM(100)) -> Dense(1, sigmoid)',
    'n_steps'       : N_STEPS,
    'seed'          : SEED,
    'fecha_export'  : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'tickers'       : {},
    'comparacion_vs_lstm': comparacion,
    'conclusion_comparacion': conclusion
}

for ticker, res in resultados.items():
    export_data['tickers'][ticker] = {
        'senal'             : res['senal'],
        'tipo_features'     : res['tipo_features'],
        'metricas'          : res['metricas'],
        'matriz_confusion'  : res['matriz_confusion'],
        'historial_epocas'  : res['historial_epocas'],
        'historico_senales' : res['historico_senales'][-60:]  # últimos 60 días para no saturar el JSON
    }

with open('datos_bilstm_clf.json', 'w', encoding='utf-8') as f:
    json.dump(export_data, f, ensure_ascii=False, indent=2)

print(f'✓ Archivo datos_bilstm_clf.json exportado ({len(resultados)} tickers)')
print(f'  Tamaño aproximado: {len(json.dumps(export_data)) / 1024:.1f} KB')

# Descargar el archivo desde Colab (opcional)
try:
    from google.colab import files
    files.download('datos_bilstm_clf.json')
    print('✓ Descarga iniciada en el navegador')
except Exception as e:
    print(f'  (Descarga automática no disponible en este entorno: {e})')

## Resultado

Las colecciones `predicciones` y `metricas_modelos` contienen resultados reales del BiLSTM (`modelo='BiLSTM'`):
- **5 tickers** procesados con la misma metodología que el LSTM (Notebook 3): mismos features,
  mismo `N_STEPS=20`, misma partición temporal 80/20 — comparación justa entre arquitecturas.
- **Bidirectional(LSTM(200)) → Dropout(0.3) → Bidirectional(LSTM(100))** → más del doble de
  parámetros que el LSTM unidireccional equivalente.
- **Comparación BiLSTM vs. LSTM** calculada ticker por ticker (Δ Accuracy, Δ F1) con una
  conclusión automática sobre si la bidireccionalidad aporta valor en este dataset.
- **class_weight balanceado**, **MinMaxScaler ajustado solo con train**, **SEED=42 fijo** —
  misma disciplina metodológica que los notebooks anteriores.
- **datos_bilstm_clf.json** exportado, incluyendo la sección `comparacion_vs_lstm` para que
  el frontend pueda mostrar ambos modelos lado a lado.

**La API** puede exponer estos resultados con un endpoint análogo a `/api/rnns/{ticker}`
filtrando `modelo='BiLSTM'` en vez de `modelo='LSTM'` (mismo patrón ya usado para SVC y LSTM).

**Pipeline completo:** `precios_ohlcv` → BiLSTM → `predicciones` + `metricas_modelos` + `datos_bilstm_clf.json` → comparación vs. LSTM ✓